# Test Notebook

Smoke test for running Jupyter notebooks against the local `fantasy-player-valuation` environment.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "analysis":
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(sys.executable)
print(repo_root)

In [ ]:
import ffvaluation
import json
import sqlite3

import pandas as pd
from ffvaluation.sources.sleeper import trade_sides_dataframe

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(ffvaluation.__file__)

# USER

In [ ]:
db_path = repo_root / "data" / "raw" / "sleeper" / "discovery" / "discovery.sqlite"
print(db_path)

if db_path.exists():
    with sqlite3.connect(db_path) as con:
        counts = {
            table: con.execute(f"select count(*) from {table}").fetchone()[0]
            for table in ["users", "frontier", "leagues", "league_users"]
        }
    counts
else:
    "discovery.sqlite not found"

In [ ]:
def find_sleeper_user(term: str, db_path=db_path):
    """Search discovery users and frontier rows by user id, username, or display name."""
    normalized = term.strip().lower()
    if not normalized:
        raise ValueError("term must not be blank")
    if not db_path.exists():
        raise FileNotFoundError(db_path)

    with sqlite3.connect(db_path) as con:
        con.row_factory = sqlite3.Row
        users = con.execute(
            """
            select user_id, display_name
            from users
            where lower(user_id) = ? or lower(display_name) = ?
            order by display_name, user_id
            """,
            (normalized, normalized),
        ).fetchall()
        frontier = con.execute(
            """
            select user_id, username, display_name, discovered_at,
                   discovered_from_league_id, expanded_at
            from frontier
            where lower(user_id) = ?
               or lower(coalesce(username, '')) = ?
               or lower(coalesce(display_name, '')) = ?
            order by discovered_at, user_id
            """,
            (normalized, normalized, normalized),
        ).fetchall()

    return {
        "users": [dict(row) for row in users],
        "frontier": [dict(row) for row in frontier],
    }


find_sleeper_user("bbroc")

# TRADES

In [15]:
trade_db_path = repo_root / "data" / "sample" / "sleeper" / "trades" / "sample.sqlite"
print(trade_db_path)

if not trade_db_path.exists():
    raise FileNotFoundError(
        f"{trade_db_path} not found. Run `ffvaluation sample-sleeper-trades` first."
    )

with sqlite3.connect(trade_db_path) as con:
    trades_df = pd.read_sql_query(
        """
        select *
        from trades
        order by created_at desc, league_id, transaction_id
        """,
        con,
    )
    trade_context_df = pd.read_sql_query(
        """
        select
            t.*,
            l.league_name,
            l.league_season,
            l.total_rosters,
            l.is_dynasty,
            l.is_superflex,
            l.ppr,
            l.te_premium,
            l.target_format_guess
        from trades t
        left join leagues l on l.league_id = t.league_id
        order by t.created_at desc, t.league_id, t.transaction_id
        """,
        con,
    )
    players_df = pd.read_sql_query(
        "select * from players order by full_name, player_id"
        if con.execute("select count(*) from sqlite_master where type = 'table' and name = 'players'").fetchone()[0]
        else "select null as player_id, null as full_name where false",
        con,
    )
    trade_player_moves_df = pd.read_sql_query(
        """
        select
            t.transaction_id,
            t.league_id,
            t.created_at,
            'add' as move_type,
            moves.key as player_id,
            moves.value as roster_id,
            p.full_name as player_name,
            p.position,
            p.team
        from trades t
        join json_each(t.adds) moves
        left join players p on p.player_id = moves.key
        where moves.key is not null
        union all
        select
            t.transaction_id,
            t.league_id,
            t.created_at,
            'drop' as move_type,
            moves.key as player_id,
            moves.value as roster_id,
            p.full_name as player_name,
            p.position,
            p.team
        from trades t
        join json_each(t.drops) moves
        left join players p on p.player_id = moves.key
        where moves.key is not null
        order by created_at desc, transaction_id, move_type, player_name
        """,
        con,
    )

player_name_by_id = dict(zip(players_df["player_id"].astype(str), players_df["full_name"]))


def readable_player_map(value, direction):
    payload = json.loads(value or "{}")
    if not isinstance(payload, dict):
        return ""
    return "; ".join(
        f"{player_name_by_id.get(str(player_id), player_id)} {direction} roster {roster_id}"
        for player_id, roster_id in payload.items()
    )


trade_context_df["adds_readable"] = trade_context_df["adds"].map(
    lambda value: readable_player_map(value, "to")
)
trade_context_df["drops_readable"] = trade_context_df["drops"].map(
    lambda value: readable_player_map(value, "from")
)

print(f"{len(trades_df):,} trades across {trades_df['league_id'].nunique():,} leagues")
trade_context_df[
    [
        "created_at",
        "league_name",
        "transaction_id",
        "roster_ids",
        "adds",
        "adds_readable",
        "drops",
        "drops_readable",
        "draft_picks",
    ]
].head()

C:\dev\fantasy_player_valuation\data\sample\sleeper\trades\sample.sqlite
1,651 trades across 92 leagues


,created_at,league_name,transaction_id,roster_ids,adds,adds_readable,drops,drops_readable,draft_picks
0,2025-12-29T18:15:13.643000+00:00,479 Dynasty League,1311431372170076160,"[8,9]","{""11600"":9,""12472"":9,""12510"":9,""4035"":8}",Ja'Tavion Sanders to roster 9; Raheim Sanders ...,"{""11600"":8,""12472"":8,""12510"":8,""4035"":9}",Ja'Tavion Sanders from roster 8; Raheim Sander...,[]
1,2025-12-29T16:50:28.994000+00:00,Transparent League,1311410045602234368,"[5,8]","{""12474"":8,""6904"":5,""8144"":8}",Woody Marks to roster 8; Jalen Hurts to roster...,"{""12474"":5,""6904"":8,""8144"":5}",Woody Marks from roster 5; Jalen Hurts from ro...,"[{""league_id"":null,""owner_id"":5,""previous_owne..."
2,2025-12-28T19:01:04.783000+00:00,479 Dynasty League,1311080523417800704,"[1,8]","{""10232"":1}",Michael Wilson to roster 1,"{""10232"":8}",Michael Wilson from roster 8,"[{""league_id"":null,""owner_id"":8,""previous_owne..."
3,2025-12-26T17:43:45.721000+00:00,alt,1310336290050301952,"[11,12]","{""11586"":11,""8167"":12,""8408"":12}",Blake Corum to roster 11; Christian Watson to ...,"{""11586"":12,""8167"":11,""8408"":11}",Blake Corum from roster 12; Christian Watson f...,"[{""league_id"":null,""owner_id"":12,""previous_own..."
4,2025-12-25T17:46:38.976000+00:00,Transparent League,1309974628873015296,"[1,9]","{""6813"":1,""7021"":9}",Jonathan Taylor to roster 1; Rico Dowdle to ro...,"{""6813"":9,""7021"":1}",Jonathan Taylor from roster 9; Rico Dowdle fro...,"[{""league_id"":null,""owner_id"":1,""previous_owne..."


In [ ]:
trade_sides_df = trade_sides_dataframe(trade_db_path)
print(f"{len(trade_sides_df):,} trade side rows")
trade_sides_df.head(10)

In [31]:
import numpy as np
import math

p1 = (5 / 14662.5)
p2 = (1 / 14662.5)
p12 = p1 + p2
p3 = 1 - p12

ans2 = (
    (1/3) * 12 * p1**1 * p2**1 * p3**2
    + (1/3) * 20 * p1**1 * p2**1 * p3**3
    + (1/3) * 30 * p1**1 * p2**1 * p3**4
)
display(1 / (12 * p1**1 * p2**1 * p3**2))
display(1 / ans2)

3586082.7384826

2083341.4448183824